In [69]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# prepare_data.py

import pandas as pd
import re
import ast
from sklearn.model_selection import train_test_split


In [4]:
def parse_extractive_summary(value):
    """
    Parse extractive_summary ke list of integers.
    Handle berbagai format: "[0,1]", "0,1", "[6,10,12]", dll.
    """
    if pd.isna(value) or value == "":
        return []

    # Jika sudah list
    if isinstance(value, list):
        return value

    str_val = str(value).strip()

    # Coba dengan ast.literal_eval (untuk format "[0, 1]")
    try:
        parsed = ast.literal_eval(str_val)
        if isinstance(parsed, list):
            return [int(x) for x in parsed]
    except:
        pass

    # Jika gagal, ekstrak semua angka dengan regex
    numbers = re.findall(r'\d+', str_val)
    return [int(n) for n in numbers]

In [5]:
def split_sentences(text):
    """
    Memecah teks menjadi kalimat dengan aman.
    """
    if not isinstance(text, str) or pd.isna(text):
        return []

    # Ganti ? dan ! dengan .
    text = text.replace('?', '.').replace('!', '.')

    # Split dengan titik
    raw_sentences = text.split('.')

    # Bersihkan dan filter kalimat kosong/terlalu pendek
    sentences = []
    for sent in raw_sentences:
        sent = sent.strip()
        # Minimal 10 karakter dan tidak hanya angka/tanda baca
        if len(sent) > 10 and not sent.isdigit():
            sentences.append(sent + '.')

    return sentences

In [56]:
def create_extractive_dataset(df, source_name=""):
    """
    Konversi dataframe ke format extractive.

    Args:
        df: DataFrame dengan kolom 'article' dan 'extractive_summary'
        source_name: nama sumber (untuk tracking)

    Returns:
        DataFrame dengan kolom 'text', 'label', 'article_idx', 'sentence_idx'
    """
    data = []
    stats = {
        'source': source_name,
        'total_articles': 0,
        'total_sentences': 0,
        'positive_labels': 0,
        'negative_labels': 0,
        'invalid_indices': 0
    }

    for idx, row in df.iterrows():
        article = row['article']
        raw_indices = parse_extractive_summary(row['extractive_summary'])

        # Pecah artikel menjadi kalimat
        sentences = split_sentences(article)
        num_sentences = len(sentences)

        if num_sentences == 0:
            continue

        # Validasi indeks (filter yang melebihi jumlah kalimat)
        valid_indices = [i for i in raw_indices if i < num_sentences]

        if len(valid_indices) != len(raw_indices):
            stats['invalid_indices'] += (len(raw_indices) - len(valid_indices))
            print(f"Warning [{source_name}]: Article {idx} has invalid indices. "
                  f"Raw: {raw_indices}, Valid: {valid_indices}")

        # Beri label untuk setiap kalimat
        for sent_idx, sentence in enumerate(sentences):
            label = 1 if sent_idx in valid_indices else 0

            data.append({
                "text": sentence,
                "label": label,
                "article_idx": idx,
                "sentence_idx": sent_idx,
                "source": source_name
            })

            stats['total_sentences'] += 1
            if label == 1:
                stats['positive_labels'] += 1
            else:
                stats['negative_labels'] += 1

        stats['total_articles'] += 1

    # Print statistik
    print(f"\n{'='*50}")
    print(f"STATISTICS: {source_name}")
    print(f"{'='*50}")
    print(f"Articles: {stats['total_articles']}")
    print(f"Total sentences: {stats['total_sentences']}")
    print(f"Positive labels: {stats['positive_labels']} ({stats['positive_labels']/max(1,stats['total_sentences'])*100:.2f}%)")
    print(f"Negative labels: {stats['negative_labels']} ({stats['negative_labels']/max(1,stats['total_sentences'])*100:.2f}%)")
    print(f"Invalid indices filtered: {stats['invalid_indices']}")

    return pd.DataFrame(data)

In [62]:
import json
import csv
import pandas as pd

def json_to_csv_single(json_file, csv_file):
    """Mengubah satu file JSON ke CSV"""
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Jika data adalah list of dictionaries
    if isinstance(data, list):
        df = pd.DataFrame(data)
    # Jika data adalah single dictionary
    elif isinstance(data, dict):
        df = pd.DataFrame([data])
    else:
        print("Format JSON tidak dikenali")
        return

    df.to_csv(csv_file, index=False, encoding='utf-8')
    print(f"Berhasil: {json_file} -> {csv_file}")

# Penggunaan
json_to_csv_single('/content/drive/MyDrive/Colab Notebooks/data project 2 (text summarization)/train_text_clean.json', '/content/drive/MyDrive/Colab Notebooks/data project 2 (text summarization)/train_text_clean.csv')
json_to_csv_single('/content/drive/MyDrive/Colab Notebooks/data project 2 (text summarization)/dev_text_clean_1071.json', '/content/drive/MyDrive/Colab Notebooks/data project 2 (text summarization)/dev_text_clean_1071.csv')

Berhasil: /content/drive/MyDrive/Colab Notebooks/data project 2 (text summarization)/train_text_clean.json -> /content/drive/MyDrive/Colab Notebooks/data project 2 (text summarization)/train_text_clean.csv
Berhasil: /content/drive/MyDrive/Colab Notebooks/data project 2 (text summarization)/dev_text_clean_1071.json -> /content/drive/MyDrive/Colab Notebooks/data project 2 (text summarization)/dev_text_clean_1071.csv


In [64]:
# ============================================
# LOAD DATA
# ============================================
print("Loading data...")
df_train_raw = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/data project 2 (text summarization)/train_text_clean.csv')
df_dev_raw = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/data project 2 (text summarization)/dev_text_clean_1071.csv')

print(f"Train raw: {len(df_train_raw)} articles")
print(f"Dev raw: {len(df_dev_raw)} articles")

Loading data...
Train raw: 5000 articles
Dev raw: 1071 articles


In [65]:
# ============================================
# KONVERSI KE FORMAT EXTRACTIVE
# ============================================

# Rename 'clean_article' to 'article' to match the function's expectation
df_train_raw = df_train_raw.rename(columns={'clean_article': 'article'})
df_dev_raw = df_dev_raw.rename(columns={'clean_article': 'article'})

print("\nConverting train data...")
train_df = create_extractive_dataset(df_train_raw, "TRAIN")

print("\nConverting dev data...")
dev_df = create_extractive_dataset(df_dev_raw, "DEV")


Converting train data...
Warning [TRAIN]: Article 100 has invalid indices. Raw: [0, 12], Valid: [0]
Warning [TRAIN]: Article 149 has invalid indices. Raw: [0, 6], Valid: [0]
Warning [TRAIN]: Article 351 has invalid indices. Raw: [0, 14], Valid: [0]
Warning [TRAIN]: Article 430 has invalid indices. Raw: [4, 15, 12, 24, 30, 28, 34, 7, 8], Valid: [4, 15, 12, 24, 30, 28, 7, 8]
Warning [TRAIN]: Article 465 has invalid indices. Raw: [0, 13], Valid: [0]
Warning [TRAIN]: Article 559 has invalid indices. Raw: [1, 7], Valid: [1]
Warning [TRAIN]: Article 822 has invalid indices. Raw: [1, 7, 18], Valid: [1, 7]
Warning [TRAIN]: Article 881 has invalid indices. Raw: [0, 12], Valid: [0]
Warning [TRAIN]: Article 982 has invalid indices. Raw: [0, 8], Valid: [0]
Warning [TRAIN]: Article 1016 has invalid indices. Raw: [0, 14], Valid: [0]
Warning [TRAIN]: Article 1036 has invalid indices. Raw: [0, 13], Valid: [0]
Warning [TRAIN]: Article 1081 has invalid indices. Raw: [1, 9], Valid: [1]
Warning [TRAIN]: 

In [66]:
def parse_extractive_summary(value):
    """
    Parse extractive_summary ke list of integers.
    Handle berbagai format: "[0,1]", "0,1", "[6,10,12]", dll.
    """
    # Ensure value is a scalar or non-Series/DataFrame before checks
    if isinstance(value, (pd.Series, pd.DataFrame)):
        # If it's a Series/DataFrame and not empty, extract the first element
        if not value.empty:
            value = value.iloc[0]
        else: # If empty Series/DataFrame, treat as empty summary
            return []

    # Now 'value' should be a scalar, a list, or np.nan.
    # The pd.isna(value) will return a single boolean.
    if pd.isna(value) or (isinstance(value, str) and value.strip() == ""):
        return []

    # If already a list
    if isinstance(value, list):
        # Ensure elements are integers, just in case
        return [int(x) for x in value]

    str_val = str(value).strip()

    # Coba dengan ast.literal_eval (untuk format "[0, 1]")
    try:
        parsed = ast.literal_eval(str_val)
        if isinstance(parsed, list):
            return [int(x) for x in parsed]
    except (ValueError, SyntaxError):
        pass # Not a valid literal, try regex

    # Jika gagal, ekstrak semua angka dengan regex
    numbers = re.findall(r'\d+', str_val)
    return [int(n) for n in numbers]

In [54]:
# This cell is a duplicate of parse_extractive_summary and is being removed to avoid confusion.
# The correct version of parse_extractive_summary is in cell Q4-CN0QUKKLB.

In [67]:
# ============================================
# SIMPAN HASIL
# ============================================
train_df.to_csv('train_extractive.csv', index=False)
dev_df.to_csv('dev_extractive_1071.csv', index=False)

print(f"\n✅ Saved:")
print(f"   train_extractive.csv: {len(train_df)} rows")
print(f"   dev_extractive_1071.csv: {len(dev_df)} rows")


✅ Saved:
   train_extractive.csv: 65547 rows
   dev_extractive_1071.csv: 13941 rows


In [68]:
# ============================================
# SAMPLE DATA
# ============================================
print("\n" + "="*50)
print("SAMPLE TRAIN DATA:")
print("="*50)
print(train_df[['text', 'label']].head(10))

print("\n" + "="*50)
print("LABEL DISTRIBUTION:")
print("="*50)
print(f"Train - Positif: {train_df['label'].sum()}, Negatif: {len(train_df) - train_df['label'].sum()}")
print(f"Dev   - Positif: {dev_df['label'].sum()}, Negatif: {len(dev_df) - dev_df['label'].sum()}")


SAMPLE TRAIN DATA:
                                                text  label
0  Puluhan petani di Polewali Mandar, Sulawesi Ba...      1
1  Ini dilakukan sebagai bentuk protes atas kiner...      0
2  Jebolnya bendungan membuat distribusi air terh...      0
3  Menurut Abdul Rasyid, petani setempat, mereka ...      1
4  Bahkan, sebagian petani menyerahkan lahannya p...      0
5  Kepala Dusun Nepo, Kecamatan Wonomulyo, Logawa...      0
6  Namun hingga kini belum ada upaya perbaikan ya...      0
7  Pemerintah beralasan karena tidak ada dana per...      0
8  Bencana kekeringan akibat kurangnya pasokan ai...      0
9  Padahal setiap kali musim tanam para petani ha...      0

LABEL DISTRIBUTION:
Train - Positif: 11260, Negatif: 54287
Dev   - Positif: 2261, Negatif: 11680
